# Mouse in vivo models: natural pregnancy, artificial decidualization (oil) and X-Mens

* **NPD:** Wang et al. 2020 - day 8 implantation sites vs inter-implantation sites, whole-uterus bulk RNA-seq,
  DEGs at fold change >= 2 and FDR <= 0.01 (Table S2).
* **AD:** Wang et al. 2020 - day 8 oil-stimulated vs unstimulated uterine horn, whole-uterus bulk RNA-seq,
  DEGs at fold change >= 2 and FDR <= 0.01 (Table S3).
* **X-Mens:** our snRNA-seq, `dec` vs `eFib`, pseudobulk, same cutoffs. The animals that supply each
  cell type (`DEC_TP`, `EFIB_TP`) and any excluded cell subtypes (`EXCLUDE_SUBTYPES`) are set in section 3.
  The same animals for both cell types give a paired design (`~animal + condition`); different animals
  give an unpaired design (`~condition`).

In X-Mens, `timepoint` identifies the biological sample (one animal per timepoint); the 10x lanes in
`sample` are summed into their animal, so lanes are never treated as replicates.

NPD and AD are whole-tissue comparisons, so their DEGs reflect changes within cells and changes in cell
composition; X-Mens compares decidual with non-decidual stromal cells.

**Outputs** (in `OUT_DIR`):
* `XMens_dec_vs_eFib_DEGs.csv`, `_all_genes.csv`, `_pseudobulks.csv` - X-Mens DEGs and the animals used
* `NPD_DEGs_vs_AD_XMens.csv`, `_stacked.svg`, `_stacked_counts.csv` - NPD DEGs classified in AD and X-Mens
* `AD_DEGs_vs_NPD_XMens.csv`, `_stacked.svg`, `_stacked_counts.csv` - AD DEGs classified in NPD and X-Mens
* `XMens_DEGs_vs_NPD_AD.csv`, `_stacked.svg`, `_stacked_counts.csv` - X-Mens DEGs classified in NPD and AD

Sections 7 and 8 also print the same-direction agreement among shared DEGs and the agreement expected
between unrelated lists, p_up(A) * p_up(B) + p_down(A) * p_down(B).

## 1. Libraries

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc
import matplotlib.pyplot as plt

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from pydeseq2.default_inference import DefaultInference

plt.rcParams["svg.fonttype"] = "none"
pd.set_option("display.width", 200)

## 2. Datasets

In [ ]:
BULK_DIR = Path("/directoryto/BulkRnaSeq")
WANG_DIR = BULK_DIR / "Wang2020"

adata = sc.read_h5ad("/directoryto/data/20261111_mouse_only_eFib_dec_final.h5ad")
print(f"X-Mens: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
print(pd.crosstab(adata.obs["timepoint"].astype(str), adata.obs["cell_type"].astype(str)).to_string())


def read_wang(table_no):
    """Read a Wang et al. 2020 supplementary DEG table (title in row 1, header in row 3)."""
    path = sorted(WANG_DIR.glob(f"*Supplementary_Table_{table_no}.xlsx"))
    if not path:
        raise FileNotFoundError(f"No *Supplementary_Table_{table_no}.xlsx in {WANG_DIR}")
    d = pd.read_excel(path[0], header=2).iloc[:, :9]
    d.columns = ["gene_symbol", "gene_id", "transcript_id", "mean_treated", "mean_control",
                 "fold_change", "pvalue", "FDR", "description"]
    d = d.dropna(subset=["gene_id"])
    d["gene_id"] = d["gene_id"].astype(str).str.split(".").str[0]
    d["direction"] = np.where(d["fold_change"] > 1, "up", "down")
    return d.set_index("gene_id")


AD = read_wang(3)
NPD = read_wang(2)
print(f"\nAD (Table S3): {len(AD)} DEGs (up {(AD.direction == 'up').sum()}, down {(AD.direction == 'down').sum()})")
print(f"NPD (Table S2): {len(NPD)} DEGs (up {(NPD.direction == 'up').sum()}, down {(NPD.direction == 'down').sum()})")

## 3. Variables

In [ ]:
# Animals that supply each cell type
DEC_TP  = ["3.5dpi", "4.5dpi_menstrual"]      # dec cells taken from these animals
EFIB_TP = ["3.5dpi", "4.5dpi_menstrual"]      # eFib cells taken from these animals
# Same list in both = paired within animal (~animal + condition); different lists = unpaired (~condition)

# Cell subtypes removed before pseudobulking (empty list = keep all)
SUBTYPE_COL = "cell_subtype"
EXCLUDE_SUBTYPES = ["decAp"]

SAMPLE_KEY = "timepoint"            # animal ("sample" = 10x lanes, summed)
CELLTYPE_COL = "cell_type"
TEST, REF = "dec", "eFib"           # log2FC > 0 = higher in decidual cells
COUNTS_LAYER = "raw_counts"
GENE_ID_COL = "gene_ids"            # var column with Ensembl IDs
MIN_CELLS = 20                      # min cells per animal x cell type pseudobulk

# DEG cutoffs - same as Wang et al. 2020
FDR, FC, MIN_CPM = 0.01, 2.0, 3.0

XMENS_LABEL = "X-Mens\ndec vs eFib"          # bar label in the plots

OUT_DIR = BULK_DIR / "SharedDEGs" / "Mouse_InVivo_Model_Comparisons"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MARKERS = ["Prl8a2", "Alpl", "Bmp2", "Wnt4", "Hand2", "Ptgs2", "Hoxa10", "Foxo1", "Pgr", "Esr1"]

PAIRED = set(DEC_TP) == set(EFIB_TP)

# Check names
tp = adata.obs[SAMPLE_KEY].astype(str)
ct = adata.obs[CELLTYPE_COL].astype(str)
st = adata.obs[SUBTYPE_COL].astype(str)
unknown = set(DEC_TP + EFIB_TP) - set(tp.unique())
assert not unknown, f"Not found in obs['{SAMPLE_KEY}']: {unknown}. Available: {sorted(tp.unique())}"
unknown = set(EXCLUDE_SUBTYPES) - set(st.unique())
assert not unknown, f"Not found in obs['{SUBTYPE_COL}']: {unknown}. Available: {sorted(st.unique())}"
assert len(DEC_TP) >= 2 and len(EFIB_TP) >= 2, "Need at least 2 animals per cell type"

# Select cells: dec from DEC_TP animals, eFib from EFIB_TP animals, minus excluded subtypes
keep = (((ct == TEST) & tp.isin(DEC_TP)) | ((ct == REF) & tp.isin(EFIB_TP))).values \
       & ~st.isin(EXCLUDE_SUBTYPES).values
sub = adata[keep]

print(f"{TEST} from: {DEC_TP}")
print(f"{REF} from: {EFIB_TP}")
print(f"Design: {'paired, ~animal + condition' if PAIRED else 'unpaired, ~condition'}")
print(f"Excluded subtypes: {EXCLUDE_SUBTYPES if EXCLUDE_SUBTYPES else 'none'}")
print("\nSubtypes kept (cell type x subtype):")
print(pd.crosstab(sub.obs[CELLTYPE_COL].astype(str), sub.obs[SUBTYPE_COL].astype(str)).to_string())
print("\nCells used (animal x cell type):")
print(pd.crosstab(sub.obs[SAMPLE_KEY].astype(str), sub.obs[CELLTYPE_COL].astype(str)).to_string())

## 4. Build pseudobulk counts (sum of raw counts per animal x cell type)

In [ ]:
X = sp.csr_matrix(sub.layers[COUNTS_LAYER])
gene_ids = pd.Index(sub.var[GENE_ID_COL].astype(str)).str.split(".").str[0]
sym_map = pd.Series(sub.var_names.astype(str), index=gene_ids)
sym_map = sym_map[~sym_map.index.duplicated()]

obs = pd.DataFrame({"animal": sub.obs[SAMPLE_KEY].astype(str).values,
                    "condition": sub.obs[CELLTYPE_COL].astype(str).values})
obs["pb_id"] = obs["animal"] + "__" + obs["condition"]
codes, pb_ids = pd.factorize(obs["pb_id"])
onehot = sp.csr_matrix((np.ones(len(codes)), (codes, np.arange(len(codes)))), shape=(len(pb_ids), len(codes)))
pb = pd.DataFrame((onehot @ X).toarray(), index=pb_ids, columns=gene_ids)
pb = pb.T.groupby(level=0).sum().T                     # merge duplicated Ensembl IDs

meta = obs.groupby("pb_id").agg(animal=("animal", "first"), condition=("condition", "first"),
                                n_cells=("animal", "size")).loc[pb.index]
small = meta.n_cells < MIN_CELLS
if small.any():
    print(f"Dropped {small.sum()} pseudobulks with < {MIN_CELLS} cells: {list(meta.index[small])}")
meta, pb = meta[~small], pb[~small]

if PAIRED:
    # keep animals that have both cell types
    both = meta.groupby("animal")["condition"].nunique()
    meta = meta[meta.animal.isin(both[both == 2].index)]
    pb = pb.loc[meta.index]
    DESIGN = "~animal + condition"
else:
    DESIGN = "~condition"

n_per = meta.condition.value_counts()
assert n_per.get(TEST, 0) >= 2 and n_per.get(REF, 0) >= 2, \
    f"Need >= 2 pseudobulks per cell type after filtering, have {n_per.to_dict()}"
meta.to_csv(OUT_DIR / "XMens_dec_vs_eFib_pseudobulks.csv")

print(f"{len(meta)} pseudobulks ({n_per.get(TEST, 0)} {TEST}, {n_per.get(REF, 0)} {REF}); design: {DESIGN}")
print(meta[["animal", "condition", "n_cells"]].to_string())

## 5. PyDESeq2: X-Mens dec vs eFib

In [ ]:
counts = pb.round().astype(int)
counts = counts.loc[:, (counts >= 10).sum(axis=0) >= 2]
m = meta[["animal", "condition"]].copy()
m["condition"] = pd.Categorical(m["condition"], categories=[REF, TEST])
m["animal"] = m["animal"].astype("category")

inference = DefaultInference(n_cpus=4)
dds = DeseqDataSet(counts=counts, metadata=m, design=DESIGN, refit_cooks=True,
                   inference=inference, quiet=True)
dds.deseq2()
ds = DeseqStats(dds, contrast=["condition", TEST, REF], inference=inference, quiet=True)
ds.summary()
res = ds.results_df.copy()
ds.lfc_shrink(coeff=f"condition[T.{TEST}]")
res["log2FoldChange_shrunk"] = ds.results_df["log2FoldChange"]

cpm = pb.div(pb.sum(axis=1), axis=0) * 1e6
for lvl in (REF, TEST):
    res[f"meanCPM_{lvl}"] = cpm.loc[meta.index[meta.condition == lvl], res.index].mean()
expressed = res[[f"meanCPM_{REF}", f"meanCPM_{TEST}"]].max(axis=1) > MIN_CPM

sig = (res["padj"] <= FDR) & expressed
res["direction"] = "ns"
res.loc[sig & (res["log2FoldChange"] >= np.log2(FC)), "direction"] = "up"
res.loc[sig & (res["log2FoldChange"] <= -np.log2(FC)), "direction"] = "down"
res.insert(0, "gene_symbol", sym_map.reindex(res.index).values)
res.index.name = "gene_id"
res = res.sort_values("padj")
res.to_csv(OUT_DIR / "XMens_dec_vs_eFib_all_genes.csv")
res[res.direction != "ns"].to_csv(OUT_DIR / "XMens_dec_vs_eFib_DEGs.csv")

print(f"X-Mens {TEST} ({', '.join(DEC_TP)}) vs {REF} ({', '.join(EFIB_TP)}), {DESIGN}: "
      f"up {(res.direction == 'up').sum()}, down {(res.direction == 'down').sum()}  "
      f"(padj <= {FDR}, |FC| >= {FC}, CPM > {MIN_CPM})")
print(res[res.gene_symbol.isin(MARKERS)][["gene_symbol", "log2FoldChange", "padj", "direction"]].round(4).to_string())

## 6. Natural pregnancy (NPD) DEGs in AD and X-Mens

Each NPD DEG is classified by its result in AD and in X-Mens. X-Mens genes are matched by Ensembl ID,
then by symbol (Wang et al. used an older annotation).

In [ ]:
npd_table = NPD[["gene_symbol", "direction", "fold_change", "pvalue", "FDR"]] \
              .rename(columns={"direction": "NPD_direction", "fold_change": "NPD_fold_change",
                               "pvalue": "NPD_pvalue", "FDR": "NPD_FDR"})

# AD: same study and IDs
w = AD.reindex(npd_table.index)
npd_table["AD_direction"] = w["direction"].fillna("ns")
npd_table["AD_fold_change"] = w["fold_change"]
npd_table["AD_FDR"] = w["FDR"]

# X-Mens: match by Ensembl ID, then by symbol
by_id = res.reindex(npd_table.index)
missing = by_id["direction"].isna()
sym_res = res.reset_index().drop_duplicates("gene_symbol").set_index("gene_symbol")
by_sym = sym_res.reindex(npd_table.loc[missing, "gene_symbol"].values)
by_sym.index = npd_table.index[missing]
cols = by_sym.columns.intersection(by_id.columns)
by_id.loc[missing, cols] = by_sym[cols]
print(f"X-Mens: {(~missing).sum()} NPD DEGs matched by Ensembl ID, {by_sym['direction'].notna().sum()} by symbol, "
      f"{by_id['direction'].isna().sum()} not in data")
npd_table["XMens_direction"] = by_id["direction"].fillna("not tested")
npd_table["XMens_log2FC"] = by_id["log2FoldChange"].astype(float)
npd_table["XMens_padj"] = by_id["padj"].astype(float)

npd_table.index.name = "gene_id"
npd_table = npd_table.sort_values(["NPD_direction", "NPD_FDR"], ascending=[False, True])
npd_table.to_csv(OUT_DIR / "NPD_DEGs_vs_AD_XMens.csv")

# Classify and plot
COMPARE_NPD = {"AD": "Artificial\ndecidualization (AD)", "XMens": XMENS_LABEL}
CATS_NPD = {
    "Shared, up in both":         "#c0392b",   # red
    "Shared, down in both":       "#f1948a",   # light red
    "Shared, opposite direction": "#4d4d4d",   # dark gray
    "NPD DEG only":               "#d0d0d0",   # light gray
}
bars = {}
for key, label in COMPARE_NPD.items():
    b = npd_table["NPD_direction"]
    s = npd_table[f"{key}_direction"].replace("not tested", "ns")
    bars[label] = {
        "Shared, up in both":         ((b == "up") & (s == "up")).sum(),
        "Shared, down in both":       ((b == "down") & (s == "down")).sum(),
        "Shared, opposite direction": ((s != "ns") & (b != s)).sum(),
        "NPD DEG only":               (s == "ns").sum(),
    }
npd_counts = pd.DataFrame(bars).T[list(CATS_NPD)]
npd_pct = npd_counts.div(npd_counts.sum(axis=1), axis=0) * 100

flat = lambda df: df.rename(index=lambda s: s.replace("\n", " "))
print(flat(npd_counts).assign(total=npd_counts.sum(axis=1).values).to_string(), "\n")
print(flat(npd_pct).round(1).to_string())
flat(npd_counts).assign(total=npd_counts.sum(axis=1).values).to_csv(OUT_DIR / "NPD_DEGs_vs_AD_XMens_stacked_counts.csv")

fig, ax = plt.subplots(figsize=(1.4 * len(npd_pct) + 3, 4.8))
x = np.arange(len(npd_pct))
bottom = np.zeros(len(npd_pct))
for cat, color in CATS_NPD.items():
    v = npd_pct[cat].values
    ax.bar(x, v, bottom=bottom, color=color, width=0.65, label=cat)
    for xi, vi, b0 in zip(x, v, bottom):
        if vi >= 3:
            ax.text(xi, b0 + vi / 2, f"{vi:.0f}%", ha="center", va="center", fontsize=8,
                    color="white" if color in ("#c0392b", "#4d4d4d") else "black")
    bottom += v
ax.set_xticks(x)
ax.set_xticklabels(npd_pct.index, fontsize=8)
ax.set_ylabel(f"% of natural pregnancy (NPD) DEGs (n = {len(npd_table)})")
ax.set_ylim(0, 100)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig.savefig(OUT_DIR / "NPD_DEGs_vs_AD_XMens_stacked.svg", bbox_inches="tight")
plt.show()
print(f"Saved {OUT_DIR / 'NPD_DEGs_vs_AD_XMens.csv'} and {OUT_DIR / 'NPD_DEGs_vs_AD_XMens_stacked.svg'}")

## 7. Artificial decidualization (AD) DEGs in NPD and X-Mens

In [ ]:
ad_table = AD[["gene_symbol", "direction", "fold_change", "pvalue", "FDR"]] \
             .rename(columns={"direction": "AD_direction", "fold_change": "AD_fold_change",
                              "pvalue": "AD_pvalue", "FDR": "AD_FDR"})

# NPD: same study and IDs
w = NPD.reindex(ad_table.index)
ad_table["NPD_direction"] = w["direction"].fillna("ns")
ad_table["NPD_fold_change"] = w["fold_change"]
ad_table["NPD_FDR"] = w["FDR"]

# X-Mens: match by Ensembl ID, then by symbol
by_id = res.reindex(ad_table.index)
missing = by_id["direction"].isna()
sym_res = res.reset_index().drop_duplicates("gene_symbol").set_index("gene_symbol")
by_sym = sym_res.reindex(ad_table.loc[missing, "gene_symbol"].values)
by_sym.index = ad_table.index[missing]
cols = by_sym.columns.intersection(by_id.columns)
by_id.loc[missing, cols] = by_sym[cols]
print(f"X-Mens: {(~missing).sum()} AD DEGs matched by Ensembl ID, {by_sym['direction'].notna().sum()} by symbol, "
      f"{by_id['direction'].isna().sum()} not in data")
ad_table["XMens_direction"] = by_id["direction"].fillna("not tested")
ad_table["XMens_log2FC"] = by_id["log2FoldChange"].astype(float)
ad_table["XMens_padj"] = by_id["padj"].astype(float)

ad_table.index.name = "gene_id"
ad_table = ad_table.sort_values(["AD_direction", "AD_FDR"], ascending=[False, True])
ad_table.to_csv(OUT_DIR / "AD_DEGs_vs_NPD_XMens.csv")
print(f"AD DEGs: {len(ad_table)} (up {(ad_table.AD_direction == 'up').sum()}, "
      f"down {(ad_table.AD_direction == 'down').sum()})\n")

# Classify and plot
COMPARE_AD = {"NPD": "Natural pregnancy\n(NPD)", "XMens": XMENS_LABEL}
CATS_AD = {
    "Shared, up in both":         "#c0392b",   # red
    "Shared, down in both":       "#f1948a",   # light red
    "Shared, opposite direction": "#4d4d4d",   # dark gray
    "AD DEG only":                "#d0d0d0",   # light gray
}
bars, agree = {}, {}
for key, label in COMPARE_AD.items():
    b = ad_table["AD_direction"]
    s = ad_table[f"{key}_direction"].replace("not tested", "ns")
    bars[label] = {
        "Shared, up in both":         ((b == "up") & (s == "up")).sum(),
        "Shared, down in both":       ((b == "down") & (s == "down")).sum(),
        "Shared, opposite direction": ((s != "ns") & (b != s)).sum(),
        "AD DEG only":                (s == "ns").sum(),
    }
    # same-direction agreement among shared DEGs, and the agreement expected between unrelated lists
    sh = s != "ns"
    n_sh, n_same = sh.sum(), (sh & (b == s)).sum()
    exp = ((b[sh] == "up").mean() * (s[sh] == "up").mean() + (b[sh] == "down").mean() * (s[sh] == "down").mean()) \
          if n_sh else np.nan
    agree[label.replace("\n", " ")] = {"shared": n_sh, "same_direction": n_same,
                                       "% same": round(100 * n_same / n_sh, 1) if n_sh else np.nan,
                                       "% expected by chance": round(100 * exp, 1)}
ad_counts = pd.DataFrame(bars).T[list(CATS_AD)]
ad_pct = ad_counts.div(ad_counts.sum(axis=1), axis=0) * 100

flat = lambda df: df.rename(index=lambda s: s.replace("\n", " "))
print(flat(ad_counts).assign(total=ad_counts.sum(axis=1).values).to_string(), "\n")
print(flat(ad_pct).round(1).to_string())
n_untested = (ad_table["XMens_direction"] == "not tested").sum()
print(f"  X-Mens 'AD DEG only': {(ad_table['XMens_direction'] == 'ns').sum()} tested but not DEG in X-Mens, "
      f"{n_untested} not detected/tested in X-Mens\n")
print("Direction agreement among shared DEGs:")
print(pd.DataFrame(agree).T.astype({"shared": int, "same_direction": int}).to_string())
flat(ad_counts).assign(total=ad_counts.sum(axis=1).values).to_csv(OUT_DIR / "AD_DEGs_vs_NPD_XMens_stacked_counts.csv")

fig, ax = plt.subplots(figsize=(1.4 * len(ad_pct) + 3, 4.8))
x = np.arange(len(ad_pct))
bottom = np.zeros(len(ad_pct))
for cat, color in CATS_AD.items():
    v = ad_pct[cat].values
    ax.bar(x, v, bottom=bottom, color=color, width=0.65, label=cat)
    for xi, vi, b0 in zip(x, v, bottom):
        if vi >= 3:
            ax.text(xi, b0 + vi / 2, f"{vi:.0f}%", ha="center", va="center", fontsize=8,
                    color="white" if color in ("#c0392b", "#4d4d4d") else "black")
    bottom += v
ax.set_xticks(x)
ax.set_xticklabels(ad_pct.index, fontsize=8)
ax.set_ylabel(f"% of artificial decidualization (AD) DEGs (n = {len(ad_table)})")
ax.set_ylim(0, 100)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig.savefig(OUT_DIR / "AD_DEGs_vs_NPD_XMens_stacked.svg", bbox_inches="tight")
plt.show()
print(f"Saved {OUT_DIR / 'AD_DEGs_vs_NPD_XMens.csv'} and {OUT_DIR / 'AD_DEGs_vs_NPD_XMens_stacked.svg'}")

## 8. X-Mens DEGs in NPD and AD

Each X-Mens DEG is classified by its result in natural pregnancy (NPD) and artificial decidualization (AD).
Wang et al. genes are matched by Ensembl ID, then by symbol. The Wang tables list DEGs only, so an X-Mens DEG
not found in a table counts as "X-Mens DEG only" (not significant, not expressed, or not matched in their data).

In [ ]:
# X-Mens DEGs, matched to each Wang table by Ensembl ID, then by symbol
xm_table = res.loc[res.direction != "ns", ["gene_symbol", "direction", "log2FoldChange", "padj"]] \
              .rename(columns={"direction": "XMens_direction", "log2FoldChange": "XMens_log2FC",
                               "padj": "XMens_padj"})

COMPARE_XM = {"NPD": "Natural pregnancy\n(NPD)", "AD": "Artificial\ndecidualization (AD)"}
WANG_TABLES = {"NPD": NPD, "AD": AD}

for key in COMPARE_XM:
    wang = WANG_TABLES[key]
    by_id = wang.reindex(xm_table.index)
    missing = by_id["direction"].isna()
    sym_w = wang.reset_index().drop_duplicates("gene_symbol").set_index("gene_symbol")
    by_sym = sym_w.reindex(xm_table.loc[missing, "gene_symbol"].values)
    by_sym.index = xm_table.index[missing]
    cols = by_sym.columns.intersection(by_id.columns)
    by_id.loc[missing, cols] = by_sym[cols]
    print(f"{key}: {(~missing).sum()} X-Mens DEGs found by Ensembl ID, {by_sym['direction'].notna().sum()} by symbol, "
          f"{by_id['direction'].isna().sum()} not in the {key} DEG list")
    xm_table[f"{key}_direction"] = by_id["direction"].fillna("ns")      # Wang tables list DEGs only
    xm_table[f"{key}_fold_change"] = by_id["fold_change"].astype(float)
    xm_table[f"{key}_FDR"] = by_id["FDR"].astype(float)

xm_table.index.name = "gene_id"
xm_table = xm_table.sort_values(["XMens_direction", "XMens_padj"], ascending=[False, True])
xm_table.to_csv(OUT_DIR / "XMens_DEGs_vs_NPD_AD.csv")
print(f"\nX-Mens DEGs: {len(xm_table)} (up {(xm_table.XMens_direction == 'up').sum()}, "
      f"down {(xm_table.XMens_direction == 'down').sum()})\n")

# Classify and plot
CATS_XM = {
    "Shared, up in both":         "#c0392b",   # red
    "Shared, down in both":       "#f1948a",   # light red
    "Shared, opposite direction": "#4d4d4d",   # dark gray
    "X-Mens DEG only":            "#d0d0d0",   # light gray
}
bars, agree = {}, {}
for key, label in COMPARE_XM.items():
    b = xm_table["XMens_direction"]
    s = xm_table[f"{key}_direction"]
    bars[label] = {
        "Shared, up in both":         ((b == "up") & (s == "up")).sum(),
        "Shared, down in both":       ((b == "down") & (s == "down")).sum(),
        "Shared, opposite direction": ((s != "ns") & (b != s)).sum(),
        "X-Mens DEG only":            (s == "ns").sum(),
    }
    # same-direction agreement among shared DEGs, and the agreement expected between unrelated lists
    sh = s != "ns"
    n_sh, n_same = sh.sum(), (sh & (b == s)).sum()
    exp = ((b[sh] == "up").mean() * (s[sh] == "up").mean() + (b[sh] == "down").mean() * (s[sh] == "down").mean()) \
          if n_sh else np.nan
    agree[label.replace("\n", " ")] = {"shared": n_sh, "same_direction": n_same,
                                       "% same": round(100 * n_same / n_sh, 1) if n_sh else np.nan,
                                       "% expected by chance": round(100 * exp, 1)}
xm_counts = pd.DataFrame(bars).T[list(CATS_XM)]
xm_pct = xm_counts.div(xm_counts.sum(axis=1), axis=0) * 100

flat = lambda df: df.rename(index=lambda s: s.replace("\n", " "))
print(flat(xm_counts).assign(total=xm_counts.sum(axis=1).values).to_string(), "\n")
print(flat(xm_pct).round(1).to_string(), "\n")
print("Direction agreement among shared DEGs:")
print(pd.DataFrame(agree).T.astype({"shared": int, "same_direction": int}).to_string())
flat(xm_counts).assign(total=xm_counts.sum(axis=1).values).to_csv(OUT_DIR / "XMens_DEGs_vs_NPD_AD_stacked_counts.csv")

fig, ax = plt.subplots(figsize=(1.4 * len(xm_pct) + 3, 4.8))
x = np.arange(len(xm_pct))
bottom = np.zeros(len(xm_pct))
for cat, color in CATS_XM.items():
    v = xm_pct[cat].values
    ax.bar(x, v, bottom=bottom, color=color, width=0.65, label=cat)
    for xi, vi, b0 in zip(x, v, bottom):
        if vi >= 3:
            ax.text(xi, b0 + vi / 2, f"{vi:.0f}%", ha="center", va="center", fontsize=8,
                    color="white" if color in ("#c0392b", "#4d4d4d") else "black")
    bottom += v
ax.set_xticks(x)
ax.set_xticklabels(xm_pct.index, fontsize=8)
ax.set_ylabel(f"% of X-Mens DEGs (n = {len(xm_table)})")
ax.set_ylim(0, 100)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig.savefig(OUT_DIR / "XMens_DEGs_vs_NPD_AD_stacked.svg", bbox_inches="tight")
plt.show()
print(f"Saved {OUT_DIR / 'XMens_DEGs_vs_NPD_AD.csv'} and {OUT_DIR / 'XMens_DEGs_vs_NPD_AD_stacked.svg'}")